# SDN DDoS model training and evaluation

This notebook trains Decision Tree, Random Forest, K-Nearest Neighbors, and Logistic Regression on controlled synthetic SDN telemetry. It reports held-out accuracy, macro precision, macro recall, macro F1, and per-record prediction latency.

**Scope note:** the dataset is synthetic lab telemetry, not a packet capture or evidence of live-network performance. Identifiers, labels, attack-family metadata, attack intensity, and attacker count are excluded from features to prevent obvious label leakage.

In [ ]:
from pathlib import Path
from time import perf_counter
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
DATASET = ROOT / 'data/processed/sdn_ddos_versatile.parquet'
RESULTS = ROOT / 'analytics/results'; RESULTS.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
print(f'Dataset: {DATASET}')
print(f'Plot directory: {RESULTS}')

In [ ]:
df = pd.read_parquet(DATASET)
metadata = json.loads(DATASET.with_suffix('.json').read_text())
print(f"Rows: {len(df):,} | Columns: {len(df.columns)} | Classes: {df['label'].nunique()}")
print(metadata['limitations'])
display(df.head(3))
display(df['label'].value_counts().sort_index().rename('records').to_frame())

plt.figure(figsize=(12, 4))
df['label'].value_counts().sort_index().plot.bar(color='#3572A5')
plt.title('Balanced controlled SDN telemetry by class'); plt.ylabel('Windows'); plt.xticks(rotation=35, ha='right'); plt.tight_layout()
plt.savefig(RESULTS / 'class_distribution.png', dpi=160); plt.show()

## Prepare an honest feature matrix

Categorical telemetry context is one-hot encoded. Numeric telemetry is median-imputed and standardized; this supports KNN and logistic regression. The split is stratified into 70% train, 15% validation, and 15% test. The validation partition is retained for later tuning; metrics below use only the held-out test set.

In [ ]:
leakage_or_id = {'timestamp_utc', 'experiment_id', 'window_id', 'window_sequence', 'src_host', 'dst_host', 'label', 'attack_family', 'data_source', 'schema_version', 'attack_intensity', 'attacker_count'}
feature_columns = [c for c in df.columns if c not in leakage_or_id]
categorical = df[feature_columns].select_dtypes(include=['object', 'category']).columns.tolist()
numeric = [c for c in feature_columns if c not in categorical]
X, y = df[feature_columns], df['label']
X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size=.30, random_state=RANDOM_STATE, stratify=y)
X_valid, X_test, y_valid, y_test = train_test_split(X_holdout, y_holdout, test_size=.50, random_state=RANDOM_STATE, stratify=y_holdout)
print(f'Features: {len(feature_columns)} ({len(numeric)} numeric, {len(categorical)} categorical)')
print(f'Train/validation/test: {len(X_train):,} / {len(X_valid):,} / {len(X_test):,}')
assert not set(leakage_or_id - {'label'}).intersection(feature_columns)
preprocessor = ColumnTransformer([('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric), ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical)])

## Train four lightweight baselines

The models reflect the literature survey’s practical ML baseline: interpretable tree, ensemble tree, distance-based classifier, and linear classifier. This is intentionally not a deep-learning benchmark.

In [ ]:
models = {'Decision Tree': DecisionTreeClassifier(max_depth=16, min_samples_leaf=3, class_weight='balanced', random_state=RANDOM_STATE), 'Random Forest': RandomForestClassifier(n_estimators=150, min_samples_leaf=2, class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE), 'KNN': KNeighborsClassifier(n_neighbors=11, weights='distance', n_jobs=-1), 'Logistic Regression': LogisticRegression(C=2.0, max_iter=1200, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)}
fitted, rows = {}, []
for name, model in models.items():
    pipeline = Pipeline([('preprocess', preprocessor), ('model', model)])
    fit_start = perf_counter(); pipeline.fit(X_train, y_train); fit_seconds = perf_counter() - fit_start
    pred_start = perf_counter(); pred = pipeline.predict(X_test); predict_seconds = perf_counter() - pred_start
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, pred, average='macro', zero_division=0)
    rows.append({'model': name, 'accuracy': accuracy_score(y_test, pred), 'precision_macro': precision, 'recall_macro': recall, 'f1_macro': f1, 'fit_seconds': fit_seconds, 'prediction_ms_per_record': 1000 * predict_seconds / len(X_test)})
    fitted[name] = (pipeline, pred)
results = pd.DataFrame(rows).sort_values(['f1_macro', 'accuracy'], ascending=False).reset_index(drop=True)
display(results.round(4))
results.to_csv(RESULTS / 'model_metrics.csv', index=False)

In [ ]:
metric_plot = results.melt(id_vars='model', value_vars=['accuracy', 'precision_macro', 'recall_macro', 'f1_macro'], var_name='metric', value_name='score')
plt.figure(figsize=(11, 5)); sns.barplot(metric_plot, x='model', y='score', hue='metric', palette='deep')
plt.ylim(0, 1.05); plt.title('Held-out test performance by model'); plt.ylabel('Macro score / accuracy'); plt.xlabel(''); plt.legend(title=''); plt.tight_layout()
plt.savefig(RESULTS / 'model_comparison.png', dpi=160); plt.show()
plt.figure(figsize=(8, 4)); sns.barplot(results, x='model', y='prediction_ms_per_record', color='#D65F5F')
plt.title('Held-out inference latency'); plt.ylabel('Milliseconds per record'); plt.xlabel(''); plt.tight_layout()
plt.savefig(RESULTS / 'inference_latency.png', dpi=160); plt.show()

## Error analysis and feature relevance

The confusion matrix uses the best macro-F1 model. Random Forest feature importance is shown separately because it provides a useful global ranking; it is descriptive rather than causal.

In [ ]:
best_name = results.loc[0, 'model']; best_pipeline, best_pred = fitted[best_name]
labels = sorted(y_test.unique()); cm = confusion_matrix(y_test, best_pred, labels=labels, normalize='true')
plt.figure(figsize=(12, 10)); sns.heatmap(cm, cmap='Blues', vmin=0, vmax=1, xticklabels=labels, yticklabels=labels, square=True)
plt.title(f'Normalized confusion matrix — {best_name}'); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(RESULTS / 'best_model_confusion_matrix.png', dpi=180); plt.show()
rf = fitted['Random Forest'][0]; feature_names = rf.named_steps['preprocess'].get_feature_names_out()
importance = pd.Series(rf.named_steps['model'].feature_importances_, index=feature_names).sort_values(ascending=False).head(18).sort_values()
plt.figure(figsize=(9, 7)); importance.plot.barh(color='#55A868'); plt.title('Random Forest top 18 feature importances'); plt.xlabel('Impurity-based importance'); plt.tight_layout()
plt.savefig(RESULTS / 'random_forest_feature_importance.png', dpi=160); plt.show()
print(f"Best held-out macro-F1: {best_name} ({results.loc[0, 'f1_macro']:.4f})")

## Conclusion

The notebook saves plots and `model_metrics.csv` to `analytics/results`. Before deployment claims, validate against separate real OpenFlow/Ryu telemetry and benchmark data, then measure the full Kafka → Spark → model → Ryu response latency.